In [ ]:
import nflreadpy as nfl

ff_rankings = nfl.load_ff_rankings()
print(ff_rankings.shape)
print(ff_rankings.columns)
print(ff_rankings.head())



In [ ]:
print(ff_rankings["ecr_type"].value_counts())
print(ff_rankings["page_type"].value_counts())
#print(ff_rankings["ecr_type"].unique())


In [ ]:
players = ff_rankings.filter(
    ff_rankings["pos"].is_in(["QB", "RB", "WR", "TE"])
)

print(players.shape)

print(
    players.select([
        "player",
        "pos",
        "team",
        "ecr",
        "best",
        "worst",
        "bye",
        "ecr_type"
    ]).head(30)
)

In [ ]:
redraft = ff_rankings.filter(
    ff_rankings["page_type"] == "redraft-overall"
)

print(redraft.shape)

print(
    redraft.select([
        "player",
        "pos",
        "team",
        "ecr",
        "best",
        "worst",
        "bye"
    ]).sort("ecr").head(50)
)

In [ ]:
draft_board = redraft.select([
    "player",
    "pos",
    "team",
    "ecr",
    "best",
    "worst",
    "bye"
]).sort("ecr")

draft_board.head(100)

In [ ]:
import polars as pl

draft_board = draft_board.with_columns(
    (pl.col("worst") - pl.col("best")).alias("expert_range")
)

draft_board.select([
    "player",
    "pos",
    "ecr",
    "best",
    "worst",
    "expert_range"
]).sort("ecr").head(50)



In [ ]:
redraft.select([
    "player", "pos", "team", "ecr", "best", "worst", "bye"
]).sort("ecr").head(30)

In [ ]:
import polars as pl

draft_board = redraft.select([
    "player",
    "pos",
    "team",
    "ecr",
    "best",
    "worst",
    "bye"
]).with_columns(
    (pl.col("worst") - pl.col("best")).alias("expert_range")
).sort("ecr")

print(
    draft_board.select([
        "player",
        "pos",
        "ecr",
        "best",
        "worst",
        "expert_range"
    ]).head(50)
)

In [ ]:
draft_board = draft_board.with_columns(
    (pl.col("expert_range") / pl.col("ecr")).alias("uncertainty")
)

draft_board.select([
    "player",
    "pos",
    "ecr",
    "best",
    "worst",
    "expert_range",
    "uncertainty"
]).sort("ecr").head(30)



In [ ]:
draft_board = draft_board.with_columns(
    (
        100 / pl.col("ecr")
        - 2 * pl.col("uncertainty")
    ).alias("draft_score")
)

draft_board.sort("draft_score", descending=True).head(50)

In [ ]:
position_rank = (
    draft_board
    .with_columns(
        pl.col("ecr")
        .rank()
        .over("pos")
        .alias("position_rank")
    )
)

In [ ]:
draft_board = (
    redraft
    .select([
        "player",
        "pos",
        "team",
        "ecr",
        "best",
        "worst",
        "bye"
    ])
    .sort("ecr")
)

print(draft_board.head(50))

In [ ]:
draft_board = draft_board.with_columns(
    (pl.col("worst") - pl.col("best")).alias("expert_range")
)

In [ ]:
draft_board = draft_board.with_columns(
    (1 / (1 + pl.col("expert_range"))).alias("confidence")
)

In [ ]:
draft_board = draft_board.with_columns(
    (
        100 - pl.col("ecr")
    ).alias("draft_score")
)

In [ ]:
draft_board = draft_board.sort(
    "draft_score",
    descending=True
)

print(
    draft_board.select([
        "player",
        "pos",
        "team",
        "ecr",
        "best",
        "worst",
        "expert_range",
        "confidence"
    ]).head(50)
)

In [ ]:
import polars as pl

def show_position(position, n=15):
    position = position.upper()

    filtered = (
        draft_board
        .filter(pl.col("pos") == position)
        .sort("ecr")
        .select([
            "player",
            "pos",
            "team",
            "ecr",
            "best",
            "worst",
            "expert_range",
            "confidence"
        ])
        .head(n)
    )

    print(f"\n===== TOP {n} {position}s =====")
    
    return filtered

In [73]:
show_position("RB")


===== TOP 15 RBs =====


player,pos,team,ecr,best,worst,expert_range,confidence
str,str,str,f64,i64,i64,i64,f64
"""Jahmyr Gibbs""","""RB""","""DET""",2.93,1,7,6,0.142857
"""Bijan Robinson""","""RB""","""ATL""",3.71,1,5,4,0.2
"""Christian McCaffrey""","""RB""","""SF""",10.22,5,31,26,0.037037
"""Jonathan Taylor""","""RB""","""IND""",12.69,7,33,26,0.037037
"""Ashton Jeanty""","""RB""","""LV""",16.62,7,34,27,0.035714
…,…,…,…,…,…,…,…
"""Kenneth Walker III""","""RB""","""KC""",28.07,13,56,43,0.022727
"""Derrick Henry""","""RB""","""BAL""",38.26,14,50,36,0.027027
"""Jeremiyah Love""","""RB""","""ARI""",40.21,26,67,41,0.02381


In [74]:
show_position("QB")


===== TOP 15 QBs =====


player,pos,team,ecr,best,worst,expert_range,confidence
str,str,str,f64,i64,i64,i64,f64
"""Josh Allen""","""QB""","""BUF""",25.9,21,37,16,0.058824
"""Lamar Jackson""","""QB""","""BAL""",33.52,25,70,45,0.021739
"""Drake Maye""","""QB""","""NE""",38.28,26,119,93,0.010638
"""Joe Burrow""","""QB""","""CIN""",46.79,27,101,74,0.013333
"""Jayden Daniels""","""QB""","""WAS""",54.35,27,108,81,0.012195
…,…,…,…,…,…,…,…
"""Brock Purdy""","""QB""","""SF""",96.48,63,130,67,0.014706
"""Jaxson Dart""","""QB""","""NYG""",97.98,59,129,70,0.014085
"""Patrick Mahomes II""","""QB""","""KC""",100.36,70,138,68,0.014493


In [75]:
show_position("DST")


===== TOP 15 DSTs =====


player,pos,team,ecr,best,worst,expert_range,confidence
str,str,str,f64,i64,i64,i64,f64
"""Houston Texans""","""DST""","""HOU""",153.71,143,176,33,0.029412
"""Denver Broncos""","""DST""","""DEN""",163.01,149,210,61,0.016129
"""Los Angeles Rams""","""DST""","""LAR""",167.31,148,186,38,0.025641
"""Seattle Seahawks""","""DST""","""SEA""",167.56,151,194,43,0.022727
"""Philadelphia Eagles""","""DST""","""PHI""",174.91,156,235,79,0.0125
…,…,…,…,…,…,…,…
"""Baltimore Ravens""","""DST""","""BAL""",199.59,171,356,185,0.005376
"""Green Bay Packers""","""DST""","""GB""",203.24,184,280,96,0.010309
"""Kansas City Chiefs""","""DST""","""KC""",210.3,151,280,129,0.007692


In [76]:
show_position("WR")


===== TOP 15 WRs =====


player,pos,team,ecr,best,worst,expert_range,confidence
str,str,str,f64,i64,i64,i64,f64
"""Ja'Marr Chase""","""WR""","""CIN""",1.51,1,6,5,0.166667
"""Puka Nacua""","""WR""","""LAR""",3.47,1,9,8,0.111111
"""Jaxon Smith-Njigba""","""WR""","""SEA""",4.77,1,11,10,0.090909
"""Amon-Ra St. Brown""","""WR""","""DET""",5.46,3,9,6,0.142857
"""CeeDee Lamb""","""WR""","""DAL""",8.84,4,18,14,0.066667
…,…,…,…,…,…,…,…
"""George Pickens""","""WR""","""DAL""",20.11,11,40,29,0.033333
"""Rashee Rice""","""WR""","""KC""",22.39,11,45,34,0.028571
"""DeVonta Smith""","""WR""","""PHI""",23.92,11,45,34,0.028571


In [70]:
show_position("TE")


===== TOP 15 TEs =====


player,pos,team,ecr,best,worst,expert_range,confidence
str,str,str,f64,i64,i64,i64,f64
"""Brock Bowers""","""TE""","""LV""",17.94,10,32,22,0.043478
"""Trey McBride""","""TE""","""ARI""",20.93,16,33,17,0.055556
"""Colston Loveland""","""TE""","""CHI""",38.13,22,87,65,0.015152
"""Tyler Warren""","""TE""","""IND""",54.39,22,85,63,0.015625
"""Harold Fannin Jr.""","""TE""","""CLE""",74.68,32,112,80,0.012346
…,…,…,…,…,…,…,…
"""Dalton Kincaid""","""TE""","""BUF""",113.72,85,155,70,0.014085
"""Jake Ferguson""","""TE""","""DAL""",114.25,73,173,100,0.009901
"""Isaiah Likely""","""TE""","""NYG""",119.07,85,236,151,0.006579


In [77]:
show_position("K")



===== TOP 15 Ks =====


player,pos,team,ecr,best,worst,expert_range,confidence
str,str,str,f64,i64,i64,i64,f64
"""Brandon Aubrey""","""K""","""DAL""",185.86,139,193,54,0.018182
"""Ka'imi Fairbairn""","""K""","""HOU""",192.4,170,212,42,0.023256
"""Cameron Dicker""","""K""","""LAC""",193.91,167,226,59,0.016667
"""Cam Little""","""K""","""JAC""",198.36,168,253,85,0.011628
"""Jason Myers""","""K""","""SEA""",200.32,171,253,82,0.012048
…,…,…,…,…,…,…,…
"""Andy Borregales""","""K""","""NE""",223.51,188,333,145,0.006849
"""Harrison Mevis""","""K""","""LAR""",225.52,192,282,90,0.010989
"""Chase McLaughlin""","""K""","""TB""",226.03,193,282,89,0.011111
